Gemini-API-Key

In [99]:
!pip install -q google-generativeai pdfplumber

In [98]:
import google.generativeai as genai
import json
import re
import pdfplumber
import time

from google.colab import userdata, files
from typing import TypedDict, List, Optional
genai.configure(api_key=userdata.get('Gemini-API-Key'))

In [97]:
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type
from google.api_core import exceptions


In [95]:
import logging
logging.getLogger('tornado.access').setLevel(logging.CRITICAL)

In [94]:
import pkg_resources
print(pkg_resources.get_distribution('google-generativeai').version)

0.8.6


In [139]:
import google.generativeai as genai
import json
import re
import pdfplumber
import time
import logging

from google.colab import userdata, files
from typing import TypedDict, List, Optional
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type
from google.api_core import exceptions

# Configure API Key
genai.configure(api_key=userdata.get('Gemini-API-Key'))

# Configure model
model = genai.GenerativeModel("gemini-2.5-flash-lite")

# Suppress tornado access logs
logging.getLogger('tornado.access').setLevel(logging.CRITICAL)


# =========================================================
# SCHEMA
# =========================================================

class Education(TypedDict):
    degree: Optional[str]
    institute: Optional[str]
    year: Optional[str]
    score: Optional[str]


class Experience(TypedDict):
    company: Optional[str]
    role: Optional[str]
    duration: Optional[str]
    highlights: List[str]


class Project(TypedDict):
    name: Optional[str]
    description: Optional[str]
    tech_stack: List[str]


class Resume(TypedDict):
    name: Optional[str]
    email: Optional[str]
    phone: Optional[str]
    linkedin_url: Optional[str]
    education: List[Education]
    skills: List[str]
    experience: List[Experience]
    projects: List[Project]
    total_experience_years: float
    summary: Optional[str]


# =========================================================
# PROMPT
# =========================================================

PROMPT = """
Extract resume data.

Rules:
1. Return valid JSON only
2. Use null for missing fields
3. Do not invent values
4. Normalize phone to +91XXXXXXXXXX
5. Skills must be clean and unique
6. Summary must be one sentence
"""


# =========================================================
# HELPER FUNCTIONS
# =========================================================

def clean_json(text):

    text = (
        text
        .replace("```json", "")
        .replace("```", "")
        .strip()
    )

    return json.loads(text)


def validate(data):

    # Email validation
    email = data.get("email")

    if email and "@" not in email:
        data["email"] = None

    # Phone normalization
    phone = data.get("phone")

    if phone:

        digits = re.sub(r"\\D", "", phone)

        if len(digits) == 10:
            data["phone"] = f"+91{digits}"

    # Experience validation
    exp = data.get("total_experience_years")

    if exp is not None and exp < 0:
        data["total_experience_years"] = 0.0

    return data


@retry(
    wait=wait_exponential(multiplier=1, min=4, max=60),
    stop=stop_after_attempt(15), # Increased retry attempts
    retry=retry_if_exception_type(exceptions.TooManyRequests)
)
def generate_content_with_retry(prompt_text, generation_config):
    return model.generate_content(prompt_text, generation_config=generation_config)


# =========================================================
# WAY 1 — PROMPT ONLY
# =========================================================

def parse_prompt(text):

    generation_config={
        "temperature": 0.2,
        "max_output_tokens": 2048
    }

    response = generate_content_with_retry(
        f"{PROMPT}\\n\\n{text}",
        generation_config=generation_config
    )

    return validate(
        clean_json(response.text)
    )


# =========================================================
# WAY 2 — MIME TYPE
# =========================================================

def parse_mime(text):

    generation_config={
        "temperature": 0.2,
        "max_output_tokens": 2048,
        "response_mime_type": "application/json"
    }

    response = generate_content_with_retry(
        f"{PROMPT}\\n\\n{text}",
        generation_config=generation_config
    )

    return validate(
        clean_json(response.text)
    )


# =========================================================
# WAY 3 — SCHEMA
# =========================================================

def parse_schema(text):

    generation_config={
        "temperature": 0.2,
        "max_output_tokens": 2048,
        "response_mime_type": "application/json",
        "response_schema": Resume
    }

    response = generate_content_with_retry(
        f"{PROMPT}\\n\\n{text}",
        generation_config=generation_config
    )

    return validate(
        clean_json(response.text)
    )


# =========================================================
# DISPLAY FUNCTION
# =========================================================

def show(title, data):

    print(f"\\n{'='*20} {title} {'='*20}")

    print(
        json.dumps(
            data,
            indent=2,
            ensure_ascii=False
        )
    )

    print("\\nSUMMARY")

    print(
        f"Name: {data.get('name')} | "
        f"Email: {data.get('email')} | "
        f"Phone: {data.get('phone')}"
    )

    print(
        "Top Skills:",
        ", ".join(data.get("skills", [])[:5])
    )

    print(
        "Experience:",
        data.get("total_experience_years"),
        "years"
    )


# =========================================================
# TEST RESUME 1
# =========================================================

resume1 = """
ANJALI SHARMA
Email: anjali.sharma@gmail.com | Phone: 9876543210
LinkedIn: linkedin.com/in/anjali-sharma

EDUCATION
- B.Tech in Computer Science, IIT Delhi, 2024 (CGPA: 8.9)

SKILLS
Python, Django, FastAPI, React.js, MongoDB

EXPERIENCE
Software Engineering Intern, Flipkart

PROJECTS
- AgriBot chatbot using Gemini API
"""


# =========================================================
# TEST RESUME 2
# =========================================================

resume2 = """
hii i am ravi (ravi.k@yahoo.in / 8765-432-100)

btech from vit vellore 2023

skills:
java, python, react

internship at zoho

project:
NoteShare app using firebase
"""


# =========================================================
# TEST 1 OUTPUT
# =========================================================

print("\\n############ TEST RESUME 1 ############")

show(
    "WAY 1 - PROMPT",
    parse_prompt(resume1)
)

show(
    "WAY 2 - MIME",
    parse_mime(resume1)
)

show(
    "WAY 3 - SCHEMA",
    parse_schema(resume1)
)


# =========================================================
# TEST 2 OUTPUT
# =========================================================

print("\\n############ TEST RESUME 2 ############")

show(
    "WAY 1 - PROMPT",
    parse_prompt(resume2)
)

show(
    "WAY 2 - MIME",
    parse_mime(resume2)
)

show(
    "WAY 3 - SCHEMA",
    parse_schema(resume2)
)


# =========================================================
# TEST 3 — PDF RESUME
# =========================================================

print("\\nUPLOAD PDF RESUME")

uploaded = files.upload()

pdf_file = list(uploaded.keys())[0]


# =========================================================
# PDF TEXT EXTRACTION
# =========================================================

def extract_text_from_pdf(path):

    text = ""

    with pdfplumber.open(path) as pdf:

        for page in pdf.pages:

            page_text = page.extract_text()

            if page_text:
                text += page_text + "\\n"

    return text


resume_text = extract_text_from_pdf(pdf_file)

print("\\nEXTRACTED TEXT:\\n")
print(resume_text)


# =========================================================
# PDF PARSING OUTPUT
# =========================================================

print("\\n############ PDF RESUME OUTPUT ############")

show(
    "WAY 1 - PROMPT",
    parse_prompt(resume_text)
)

show(
    "WAY 2 - MIME",
    parse_mime(resume_text)
)

show(
    "WAY 3 - SCHEMA",
    parse_schema(resume_text)
)


\n############ TEST RESUME 1 ############
\n==================== WAY 1 - PROMPT ====================
{
  "name": "ANJALI SHARMA",
  "email": "anjali.sharma@gmail.com",
  "phone": "+919876543210",
  "linkedin": "linkedin.com/in/anjali-sharma",
  "education": [
    {
      "degree": "B.Tech in Computer Science",
      "institution": "IIT Delhi",
      "graduation_year": 2024,
      "cgpa": 8.9
    }
  ],
  "skills": [
    "Python",
    "Django",
    "FastAPI",
    "React.js",
    "MongoDB"
  ],
  "experience": [
    {
      "title": "Software Engineering Intern",
      "company": "Flipkart",
      "years": null
    }
  ],
  "projects": [
    {
      "name": "AgriBot chatbot using Gemini API",
      "description": null
    }
  ],
  "summary": null
}
\nSUMMARY
Name: ANJALI SHARMA | Email: anjali.sharma@gmail.com | Phone: +919876543210
Top Skills: Python, Django, FastAPI, React.js, MongoDB
Experience: None years
\n==================== WAY 2 - MIME ====================
{
  "name": "ANJALI SH

Saving Hellopdf.pdf to Hellopdf.pdf
\nEXTRACTED TEXT:\n
Hello, I am Karthik Sharma (karthik.sharma@gmail.com / 9123-456-781). I
recently completed my B.Tech in Computer Science from SRM University
Chennai (2024, CGPA 8.5). I have knowledge of JavaScript, Python, and SQL
along with some experience in web development and cloud computing. I
completed a 5-month internship at Infosys as a Software Intern, where I
worked on API integration and testing. I also worked at a startup called
TechNova for 7 months as a Full Stack Developer and helped develop their
customer management dashboard. I enjoy basketball and photography in my
free time. I also created a project called StudySync — a collaborative study
platform for students built using React, Node.js, and MongoDB.\n
\n############ PDF RESUME OUTPUT ############
\n==================== WAY 1 - PROMPT ====================
{
  "name": "Karthik Sharma",
  "email": "karthik.sharma@gmail.com",
  "phone": "+919123456781",
  "summary": "Recent B.Tec